
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab — Build a Feature Engineering Pipeline

In this lab, you will build a complete feature engineering pipeline using the **CDC Diabetes Health Indicators** dataset. You will load and clean the data, create a Spark ML pipeline that handles missing values, encodes categorical features, and scales numerical features, then apply it consistently to both training and test sets. Finally, you will prepare the target column and save the pipeline for future reuse.

**Learning Objectives**

*By the end of this lab, you will be able to:*

1. Clean a raw dataset for Spark ML by casting column types, dropping columns with excessive missing values, and removing invalid and extreme values.
1. Save the cleaned data as a silver Delta table in Unity Catalog.
1. Split a dataset into training and testing sets and persist each split as a Delta table.
1. Assemble a Spark ML pipeline that imputes, indexes, one-hot encodes, scales continuous features, and vectorizes features.
1. Fit the pipeline on the training set only and apply it to both sets to prevent data leakage.
1. Prepare a numeric label column for model training.
1. Save a fitted pipeline and reload it for reuse.

**Lab Tasks**

* **Task 1:** Load Dataset and Data Preparation
  * **1.1.** Load Dataset
  * **1.2.** Data Preparation — Type Casting, Missing Columns, Outlier Removal, Save Silver Table
* **Task 2:** Split Dataset into Training and Testing Sets
* **Task 3:** Create Feature Engineering Pipeline
  * **3.1.** Analyze Data Types and Missing Values
  * **3.2.** Define Pipeline Stages and Build Pipeline
* **Task 4:** Fit the Pipeline
* **Task 5:** Transform Datasets and Prepare the Target Column
* **Task 6:** Save and Load Pipeline

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version ML V6</strong> - <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;" target="_blank">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless ML V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## ⚠️ SERVERLESS: RESTART YOUR SESSION IF NEEDED

<div style="border-left: 4px solid #FF9800; background: #FFF3E0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #E65100; font-size: 1.1em;">Restart the Serverless Session</strong>
<p style="margin: 8px 0 0 0; color: #333;">Because this notebook runs on <strong>Serverless</strong>, you may occasionally need to start a fresh session for state to reset cleanly. To do this, either:</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Restart the session</strong> — open the compute drop-down (top right) and choose <strong>New session</strong> to begin a new session, <em>or</em></li>
<li><strong>Terminate and start</strong> — <strong>terminate</strong> the current Serverless compute, then <strong>start</strong> it again before re-running the cells.</li>
</ul>
</div>
</div>
</div>

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">If you hit this error</strong>
<p style="margin: 8px 0 0 0;"><code style="background: #FCE4EC; color: #B71C1C; padding: 2px 6px; border-radius: 3px;">[CONNECT_ML.MODEL_SIZE_OVERFLOW_EXCEPTION] Spark Connect ML error: The fitted or loaded model size is about … bytes. Fit or load a model smaller than 268435456 bytes. SQLSTATE: 54000</code></p>
<p style="margin: 12px 0 0 0; color: #333;"><strong>Fix:</strong> <strong>Restart the session</strong> (compute drop-down → <strong>New session</strong>) <em>or</em> <strong>terminate and start</strong> the Serverless compute, then re-run the notebook cells from the top.</p>
</div>
</div>
</div>

## Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this lab

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-2.7

**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains **variables such as your username, catalog name, schema name, working directory, and dataset locations**. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Task 1: Load Dataset and Data Preparation

In this task, you will load the **CDC Diabetes Health Indicators** dataset and prepare it for machine learning. This health survey dataset contains demographic information and health indicators about respondents, including whether they have been diagnosed with diabetes.

You will:
- Load the raw CSV file into a Spark DataFrame
- Perform initial data preparation: type casting, removing columns with excessive missing values, and filtering outliers
- Save the cleaned dataset as a **Delta silver table** for downstream use

### Task 1.1: Load the Dataset

Load the CDC diabetes dataset from the provided path. Use the following options:
- `.option("nullValue", "null")` — read the string `"null"` as a SQL `null`
- `header="true"` — the CSV file includes a header row
- `inferSchema="true"` — let Spark automatically detect column data types
- `multiLine="true"` — handle multi-line CSV fields correctly

Once loaded, display the DataFrame to inspect its structure and column types.

In [0]:
## Set the path of the dataset
dataset_path = f"{DA.paths.datasets.cdc_diabetes}/cdc-diabetes/diabetes_binary_5050_raw.csv"

## Read the CSV file using Spark
## Set the header, inferSchema, multiLine, and nullValue options
cdc_df = <FILL_IN>

## Display the resulting DataFrame
<FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1.1: Load Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Set the path of the dataset
dataset_path = f"{DA.paths.datasets.cdc_diabetes}/cdc-diabetes/diabetes_binary_5050_raw.csv"

# Read the CSV file using Spark
cdc_df = spark.read.option("nullValue", "null").csv(
    dataset_path,
    header="true",
    inferSchema="true",
    multiLine="true",
    escape='"'
)

# Display the resulting DataFrame
display(cdc_df)
```

</div>
</details>

### Task 1.2: Data Preparation

With the data loaded, the next step is to prepare it for modeling. You will:
- **Cast data types** — Convert integer and boolean columns to `double` for compatibility with Spark ML
- **Remove columns with too many missing values** — Drop any column where more than 60% of values are missing
- **Remove outliers** — Filter records with invalid or extreme values
- **Save the cleaned data** — Write to a Delta silver table for reuse

**1.2a — Convert Data Types**

The final feature vector that a Spark ML model reads must be numeric. In this dataset every survey answer is already a number, so you only need to cast any `IntegerType` or `BooleanType` columns to `DoubleType` for consistency. (String columns would instead go through `StringIndexer` in the pipeline.)

> Print the schema after casting to verify the changes.

In [0]:
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col

## Get a list of integer and boolean columns
integer_cols = <FILL_IN>

## Cast each to double for Spark ML compatibility
for column in integer_cols:
    cdc_df = cdc_df.withColumn(column, <FILL_IN>)

## Print the schema to verify the changes
cdc_df.<FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1.2a: Convert Data Types
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col

# Get a list of integer and boolean columns
integer_cols = [
    c.name for c in cdc_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

# Cast each to double for Spark ML compatibility
for column in integer_cols:
    cdc_df = cdc_df.withColumn(column, col(column).cast("double"))

# Print the schema to verify the changes
cdc_df.printSchema()
```

</div>
</details>

**1.2b — Remove Columns with Too Many Missing Values**

Columns with a very high proportion of missing values are unlikely to be useful for modeling and can cause issues during pipeline fitting. Count the missing values in each column, then drop any column where more than **60%** of rows are null. Never drop the target column `Diabetes_binary`. The 60% threshold is a choice for this lab, not a universal rule.

> Display the missing value counts before dropping, then display the cleaned DataFrame after.

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

## Count missing values per column
missing_counts = cdc_df.agg(*[
    <FILL_IN>
    for c in cdc_df.columns
]).first().asDict()

## Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    <FILL_IN>,
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

## Set threshold: drop columns with more than 60% missing data
per_thresh = 0.6
N = cdc_df.<FILL_IN>
to_drop_missing = <FILL_IN>

print(f"Dropping columns with >{per_thresh * 100}% missing data: {to_drop_missing}")
cdc_no_missing_df = cdc_df.drop(*to_drop_missing)
display(cdc_no_missing_df)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1.2b: Remove Missing-Heavy Columns
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql.functions import col, when, sum as spark_sum

# Count missing values per column
missing_counts = cdc_df.agg(*[
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cdc_df.columns
]).first().asDict()

# Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    [(c, int(v)) for c, v in missing_counts.items()],
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

# Set threshold: drop columns with more than 60% missing data
per_thresh = 0.6
N = cdc_df.count()

to_drop_missing = [
    c for c, v in missing_counts.items()
    if v / N > per_thresh and c != "Diabetes_binary"  # never drop the target
]

print(f"Dropping columns with >{per_thresh * 100}% missing data: {to_drop_missing}")

cdc_no_missing_df = cdc_df.drop(*to_drop_missing)
display(cdc_no_missing_df)
```

</div>
</details>

**1.2c — Remove Outliers**

Before modeling, remove records with values that are invalid or outside the range chosen for this lab:
- `MentHlth` — the number of days of poor mental health in the past 30 days, so valid values are `0` to `30`. Negative values are **invalid** (data errors).
- `BMI` — body mass index. Values above 50 are **extreme but possible**. This lab removes them as a dataset-specific choice; in a real project, decide this with domain experts and document it, because filtering changes the population the model learns from.

**Keep rows where these columns are null.** In Spark, a comparison with null is not true, so a filter such as `col("MentHlth") >= 0` silently drops every row where `MentHlth` is missing. Those rows should stay, so that the pipeline's `Imputer` can fill them in. Check for nulls explicitly with `isNull()`.

Apply both filters in a single step, then print the row count before and after.

In [0]:
## Define cutoff values
MentHlth_min, MentHlth_max = 0, 30   ## days in the past 30 days
BMI_cutoff = 50                       ## upper limit chosen for this lab

## Apply both filters in a single step, keeping rows where the value is null
cdc_no_outliers_df = cdc_no_missing_df.filter(
    <FILL_IN>
)

## Print count before and after
print(<FILL_IN>)

## Display the filtered DataFrame
<FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1.2c: Remove Outliers
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Define cutoff values
MentHlth_min, MentHlth_max = 0, 30   # days in the past 30 days
BMI_cutoff = 50                       # upper limit chosen for this lab

# Apply both filters in a single step, keeping rows where the value is null
# (a comparison with null is not true, so nulls must be kept explicitly)
cdc_no_outliers_df = cdc_no_missing_df.filter(
    (col("MentHlth").isNull() | col("MentHlth").between(MentHlth_min, MentHlth_max))
    & (col("BMI").isNull() | (col("BMI") <= BMI_cutoff))
)

# Print count before and after
print(f"Row count — Before: {cdc_no_missing_df.count()} / After: {cdc_no_outliers_df.count()}")

# Display the filtered DataFrame
display(cdc_no_outliers_df)
```

</div>
</details>

**1.2d — Save the Cleaned Dataset as a Silver Table**

Save the cleaned DataFrame as a Delta table. This creates a **silver table** — a cleaned, structured version of the raw data — that can be reused in downstream tasks without re-running the data preparation steps.

In [0]:
cdc_df_full = "cdc_df_full"  ## use a fully qualified name: catalog.schema.table

## Build the silver table name
cdc_df_full_silver = <FILL_IN>

## Save as Delta table (overwrite if exists)
cdc_no_outliers_df.write.mode("overwrite").option("mergeSchema", True).<FILL_IN>

print(f"Saved silver table: {cdc_df_full_silver}")
display(cdc_no_outliers_df)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 1.2d: Save Silver Table
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Base name for the dataset
cdc_df_full = "cdc_df_full"

# Build the silver table name
cdc_df_full_silver = f"{DA.catalog_name}.{DA.schema_name}.{cdc_df_full}_silver"

# Save as Delta table (overwrite if exists)
cdc_no_outliers_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable(cdc_df_full_silver)

print(f"Saved silver table: {cdc_df_full_silver}")

# Display the resulting DataFrame
display(cdc_no_outliers_df)
```

</div>
</details>

## Task 2: Split Dataset into Training and Testing Sets

Split the cleaned dataset into training and testing sets using an approximately **80/20 split** (`randomSplit` proportions are approximate). This ensures that:
- The pipeline is **fitted only on training data** (preventing data leakage)
- The test set remains unseen during pipeline fitting, for an unbiased evaluation

After splitting, save both sets as Delta tables for reproducibility.

In [0]:
## Split the dataset: 80% training, 20% testing
train_df, test_df = cdc_no_outliers_df.<FILL_IN>

## Save each split as a Delta table
train_df.write.mode("overwrite").option("overwriteSchema", True).<FILL_IN>
test_df.write.mode("overwrite").option("overwriteSchema", True).<FILL_IN>

print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 2: Split Dataset
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Split the dataset: 80% training, 20% testing
train_df, test_df = cdc_no_outliers_df.randomSplit([0.8, 0.2], seed=42)

# Save each split as a Delta table
train_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(
    f"{DA.catalog_name}.{DA.schema_name}.cdc_df_train"
)

test_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(
    f"{DA.catalog_name}.{DA.schema_name}.cdc_df_test"
)

# Print row counts
print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")
```

</div>
</details>

## Task 3: Create Feature Engineering Pipeline

Now you will build a **Spark ML Pipeline** that automates the full feature transformation process. A pipeline ensures that every transformation learned from training data is applied identically to new data — this is the foundation of reproducible, production-ready ML workflows.

**The pipeline will include the following stages:**

| Step | Transformer | Purpose |
|------|------------|---------|
| 1 | `StringIndexer` | Convert string categories to numeric indices (categorical branch) |
| 2 | `Imputer` | Fill missing numerical values with the training-set median (numeric branch) |
| 3 | `VectorAssembler` | Combine the continuous numerical columns into a single vector (numeric branch) |
| 4 | `RobustScaler` | Scale the continuous features, robust to outliers (numeric branch) |
| 5 | `OneHotEncoder` | Convert categorical indices to binary sparse vectors (categorical branch) |
| 6 | `VectorAssembler` | Combine the scaled continuous vector, the unscaled binary columns, and any categorical vectors into the final `features` vector |

> **Note about this dataset:** every column in the CDC dataset is numeric (survey answers are stored as numeric codes), so there are no string columns. The categorical stages (1 and 5) are therefore added **only when string columns exist**. The same code then works for datasets that do have categorical columns, such as the telco data in the demos.

### Task 3.1: Analyze Data Types and Missing Values

Before building the pipeline, analyze the training set to understand which columns need which treatment:

- Cast any remaining `IntegerType` or `BooleanType` columns to `double` (apply to both `train_df` and `test_df`). After Task 1.2a this should find none; the check keeps the code safe for other datasets.
- Identify **categorical (string) columns** — these will go through `StringIndexer` → `OneHotEncoder`
- Identify **numerical (double) columns** — these will be imputed
- Split the numeric columns into **binary** 0/1 answers and **continuous** measures. Only the continuous columns are scaled. Many binary columns are mostly 0, so their IQR is 0, and `RobustScaler` would turn every value into 0 and the model would lose the feature. Binary columns are already on a 0–1 scale, so they do not need scaling. The binary columns are listed for you from the dataset's data dictionary. The list is compared in lowercase, because some column names in the raw file use a different case (for example, `hvyalcoholconsump` and `income`).
- Identify **which numeric columns have missing values** — only those need imputation

> **Important:** Exclude the target column `Diabetes_binary` from the feature columns. It is our prediction target, not an input feature.

In [0]:
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when

## Cast any remaining integer and boolean columns to double
## Apply to BOTH train_df and test_df
integer_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]
for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast(<FILL_IN>))
    test_df = test_df.withColumn(column, col(column).cast(<FILL_IN>))

## Define the target column — exclude it from all feature lists
target_col = "Diabetes_binary"

## Identify string (categorical) columns — excluding target
string_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]

## Identify numeric columns — excluding target
num_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]

## Binary 0/1 survey answers (from the data dictionary) are imputed but not scaled
## Compare in lowercase: some column names in the raw file use a different case (for example, "hvyalcoholconsump")
binary_candidates = {"highbp", "highchol", "cholcheck", "smoker", "stroke", "heartdiseaseorattack",
                     "physactivity", "fruits", "veggies", "hvyalcoholconsump", "anyhealthcare",
                     "nodocbccost", "diffwalk", "sex"}
binary_cols = [c for c in num_cols if c.lower() in binary_candidates]
continuous_cols = [c for c in num_cols if c not in binary_cols]

## Find numeric columns with missing values (only these need Imputer)
num_missing_values_logic = [count(when(col(c).isNull(), c)).alias(c) for c in num_cols]
row_dict_num = train_df.select(num_missing_values_logic).first().<FILL_IN>
num_missing_cols = [c for c in row_dict_num if row_dict_num[c] > 0]

print(f"Categorical (string) columns: {string_cols}")
print(f"Numeric columns: {num_cols}")
print(f"Binary columns (not scaled): {binary_cols}")
print(f"Continuous columns (scaled): {continuous_cols}")
print(f"Numeric columns with missing values: {num_missing_cols}")


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 3.1: Analyze Data Types and Missing Values
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when

# Cast any remaining integer and boolean columns to double
integer_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast("double"))
    test_df = test_df.withColumn(column, col(column).cast("double"))

# Define the target column — exclude it from all feature lists
target_col = "Diabetes_binary"

# Identify string (categorical) columns — excluding target
string_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, StringType) and c.name != target_col
]

# Identify numeric columns — excluding target
num_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, DoubleType) and c.name != target_col
]

# Binary 0/1 survey answers (from the data dictionary) are imputed but not scaled
# Compare in lowercase: some column names in the raw file use a different case (for example, "hvyalcoholconsump")
binary_candidates = {"highbp", "highchol", "cholcheck", "smoker", "stroke", "heartdiseaseorattack",
                     "physactivity", "fruits", "veggies", "hvyalcoholconsump", "anyhealthcare",
                     "nodocbccost", "diffwalk", "sex"}
binary_cols = [c for c in num_cols if c.lower() in binary_candidates]
continuous_cols = [c for c in num_cols if c not in binary_cols]

# Find numeric columns with missing values
num_missing_values_logic = [
    count(when(col(c).isNull(), c)).alias(c)
    for c in num_cols
]

row_dict_num = train_df.select(num_missing_values_logic).first().asDict()

num_missing_cols = [
    c for c in row_dict_num
    if row_dict_num[c] > 0
]

print(f"Categorical (string) columns: {string_cols}")
print(f"Numeric columns: {num_cols}")
print(f"Binary columns (not scaled): {binary_cols}")
print(f"Continuous columns (scaled): {continuous_cols}")
print(f"Numeric columns with missing values: {num_missing_cols}")
```

</div>
</details>

### Task 3.2: Define Pipeline Stages and Build the Pipeline

Using the column lists you identified in Task 3.1, define each pipeline stage and assemble them into a `Pipeline`.

Follow these steps in order:
1. **`StringIndexer`** — only if `string_cols` is not empty: convert string category columns to numeric indices. Use `handleInvalid="keep"` so null and unseen values are treated as a separate category rather than causing an error.
2. **`Imputer`** — only if `num_missing_cols` is not empty: fill missing values with the `median` strategy (robust to the skewed health measures in this dataset). Write the results to new columns with the `_imputed` suffix (for example, `MentHlth_imputed`).
3. **`VectorAssembler`** — assemble the **continuous** columns into a single vector called `numerical_assembled`. Use the `_imputed` version of each column that was imputed, and the original column otherwise.
4. **`RobustScaler`** — scale the assembled continuous vector. `RobustScaler` uses the interquartile range (IQR) and is less sensitive to remaining outliers than `StandardScaler`.
5. **`OneHotEncoder`** — only if `string_cols` is not empty: convert the indexed categorical columns into binary sparse vectors. Use `handleInvalid="keep"`.
6. **`VectorAssembler`** (final) — combine the scaled continuous vector, the binary columns (the `_imputed` version where one exists), and any one-hot encoded categorical features into a single `features` column.
7. **`Pipeline`** — pass all stages to a `Pipeline` in the correct order.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, RobustScaler

## Build the list of stages; some stages are added only when they are needed
stages_list = []

## Step 1: StringIndexer — only if there are string columns
categorical_cols_indexed = [c + "_index" for c in string_cols]
if string_cols:
    string_indexer = StringIndexer(inputCols=<FILL_IN>, outputCols=<FILL_IN>, handleInvalid=<FILL_IN>)
    stages_list.append(string_indexer)

## Step 2: Imputer — only if there are numeric columns with missing values
## Write the imputed values to new columns with the "_imputed" suffix, using the median
imputed_cols = [c + "_imputed" for c in num_missing_cols]
if num_missing_cols:
    imputer = Imputer(inputCols=<FILL_IN>, outputCols=<FILL_IN>, strategy=<FILL_IN>)
    stages_list.append(imputer)

## Step 3: VectorAssembler — continuous columns only; use the imputed column where one exists
imputed_map = dict(zip(num_missing_cols, imputed_cols))
numeric_inputs = [imputed_map.get(c, c) for c in continuous_cols]
binary_inputs = [imputed_map.get(c, c) for c in binary_cols]   ## not scaled
numerical_assembler = VectorAssembler(inputCols=<FILL_IN>, outputCol=<FILL_IN>)
stages_list.append(numerical_assembler)

## Step 4: RobustScaler — scale the continuous vector
numerical_scaler = RobustScaler(inputCol=<FILL_IN>, outputCol=<FILL_IN>)
stages_list.append(numerical_scaler)

## Step 5: OneHotEncoder — only if there are string columns
ohe_cols = [c + "_ohe" for c in string_cols]
if string_cols:
    one_hot_encoder = OneHotEncoder(inputCols=<FILL_IN>, outputCols=<FILL_IN>, handleInvalid=<FILL_IN>)
    stages_list.append(one_hot_encoder)

## Step 6: Final VectorAssembler — combine all features into one vector
feature_cols = ["numerical_scaled"] + binary_inputs + ohe_cols   ## ohe_cols is [] when there are no string columns
vector_assembler = VectorAssembler(inputCols=<FILL_IN>, outputCol=<FILL_IN>)
stages_list.append(vector_assembler)

## Step 7: Build the Pipeline with all stages in order
pipeline = <FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 3.2: Define Pipeline Stages
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, RobustScaler

# Define the target column
target_col = "Diabetes_binary"

# ---- Build stages list conditionally ----
stages_list = []

# Step 1: StringIndexer — only if there are string columns
categorical_cols_indexed = [c + "_index" for c in string_cols]
if string_cols:
    string_indexer = StringIndexer(
        inputCols=string_cols,
        outputCols=categorical_cols_indexed,
        handleInvalid="keep"
    )
    stages_list.append(string_indexer)

# Step 2: Imputer — only if there are numeric columns with missing values
# Write the imputed values to new columns with the "_imputed" suffix, using the median
imputed_cols = [c + "_imputed" for c in num_missing_cols]
if num_missing_cols:
    imputer = Imputer(
        inputCols=num_missing_cols,
        outputCols=imputed_cols,
        strategy="median"
    )
    stages_list.append(imputer)

# Step 3: VectorAssembler — continuous columns only; use the imputed column where one exists
imputed_map = dict(zip(num_missing_cols, imputed_cols))
numeric_inputs = [imputed_map.get(c, c) for c in continuous_cols]
binary_inputs = [imputed_map.get(c, c) for c in binary_cols]   # not scaled
numerical_assembler = VectorAssembler(
    inputCols=numeric_inputs,
    outputCol="numerical_assembled"
)
stages_list.append(numerical_assembler)

# Step 4: RobustScaler — scale the continuous vector
numerical_scaler = RobustScaler(
    inputCol="numerical_assembled",
    outputCol="numerical_scaled"
)
stages_list.append(numerical_scaler)

# Step 5: OneHotEncoder — only if there are categorical columns
ohe_cols = [c + "_ohe" for c in string_cols]
if string_cols:
    one_hot_encoder = OneHotEncoder(
        inputCols=categorical_cols_indexed,
        outputCols=ohe_cols,
        handleInvalid="keep"
    )
    stages_list.append(one_hot_encoder)

# Step 6: Final VectorAssembler — combine all available features
feature_cols = ["numerical_scaled"] + binary_inputs + ohe_cols  # ohe_cols will be [] if no string cols
vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
stages_list.append(vector_assembler)

# Step 7: Build the Pipeline
pipeline = Pipeline(stages=stages_list)
```

</div>
</details>

## Task 4: Fit the Pipeline

Fit the pipeline on the **training dataset**. During fitting, each Estimator stage (e.g., `StringIndexer`, `Imputer`, `RobustScaler`) learns parameters from `train_df` — such as category mappings, median values for imputation, and scaling factors. These learned parameters are stored in the resulting `PipelineModel`.

> **Why fit on training data only?** Statistics computed from the test set (e.g., test means used for imputation) would introduce data leakage, making your evaluation results unreliable.

In [0]:
# Fit the pipeline on the training dataset
pipeline_model = <FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 4: Fit the Pipeline
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Fit the pipeline on the training dataset
pipeline_model = pipeline.fit(train_df)
```

</div>
</details>

## Task 5: Transform Datasets and Prepare the Target Column

Apply the fitted pipeline to both datasets, then prepare the target column for modeling.

**5.1 — Transform datasets:** Use `pipeline_model.transform()` to apply all learned transformations to `train_df` and `test_df`. The result will include the final `features` vector column.

**5.2 — Prepare the target column:** Spark ML models require the target label to be a numeric column named `label`. The `Diabetes_binary` column already contains `0.0` (no diabetes) and `1.0` (diabetes). Create a `label` column from it in both the training and test transformed DataFrames, and keep only the `features` and `label` columns, so that the original target column cannot accidentally be used as a feature later.

> Display the final result showing both `features` and `label` to confirm the dataset is ready for modeling.

In [0]:
## Transform both datasets using the fitted pipeline
train_transformed_df = pipeline_model.<FILL_IN>
test_transformed_df = <FILL_IN>

## Create the 'label' column from 'Diabetes_binary' (already 0.0 / 1.0)
from pyspark.sql.functions import col
train_prepared_df = train_transformed_df.withColumn(<FILL_IN>).select(<FILL_IN>)
test_prepared_df = test_transformed_df.withColumn(<FILL_IN>).select(<FILL_IN>)

## Display features and label from the training set
display(train_prepared_df.select(<FILL_IN>))


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 5: Transform and Prepare Target
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Transform both datasets using the fitted pipeline
train_transformed_df = pipeline_model.transform(train_df)
test_transformed_df = pipeline_model.transform(test_df)

# Create the 'label' column from 'Diabetes_binary' (already 0.0 / 1.0)
from pyspark.sql.functions import col

train_prepared_df = train_transformed_df.withColumn("label", col("Diabetes_binary")).select("features", "label")
test_prepared_df = test_transformed_df.withColumn("label", col("Diabetes_binary")).select("features", "label")

# Display features and label from the training set
display(train_prepared_df.select("features", "label"))
```

</div>
</details>

## Task 6: Save and Load the Pipeline

Saving the fitted pipeline model allows you to reuse the exact same transformations in future sessions or production deployments — without re-fitting from scratch. In this task you will save the pipeline to a versioned folder in your working directory (a Unity Catalog volume), then load it back and inspect its stages.

**6a — Save the Pipeline**

In [0]:
## Save the fitted pipeline model to the working directory
pipeline_model.<FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 6a: Save Pipeline
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Save the fitted pipeline model to the working directory
pipeline_model.write().overwrite().save(f"{DA.paths.working_dir}/spark_pipelines/cdc_diabetes_v1")
print(f"Pipeline saved to: {DA.paths.working_dir}/spark_pipelines/cdc_diabetes_v1")
```

</div>
</details>

**6b — Load the Saved Pipeline and Inspect Its Stages**

In [0]:
from pyspark.ml import PipelineModel

## Load the saved pipeline model
loaded_pipeline = <FILL_IN>

## Display the pipeline stages to confirm the transformation order
<FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Task 6b: Load Pipeline
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.ml import PipelineModel

# Load the saved pipeline model
loaded_pipeline = PipelineModel.load(f"{DA.paths.working_dir}/spark_pipelines/cdc_diabetes_v1")

# Display the pipeline stages to confirm the transformation order
loaded_pipeline.stages
```

</div>
</details>

## Conclusion

In this lab, you built an end-to-end feature engineering pipeline for the CDC Diabetes Health Indicators dataset using Spark ML.

You performed key data preparation steps, including:
- Casting columns to appropriate data types  
- Removing columns with excessive missing values  
- Filtering out invalid or outlier values  

You then split the dataset into training and test sets and constructed a reusable Spark ML pipeline that:
- Adds `StringIndexer` and `OneHotEncoder` stages only when the data has categorical columns (the CDC data has none)  
- Imputes missing numerical values with the training-set median using `Imputer`  
- Scales the continuous features using `RobustScaler`, and keeps the binary 0/1 columns unscaled so that no feature is lost  
- Assembles all features into a single vector for modeling  

You also prepared the target variable by converting `Diabetes_binary` into a numeric `label` column compatible with Spark ML models. Finally, you saved the pipeline for reuse, enabling consistent and reproducible data transformations.

These core practices—structured data preparation, leakage-free transformations, and reusable pipelines—form the foundation for building reliable machine learning workflows in Databricks.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>
